# Natural language to SQL

<!-- source: https://www.youtube.com/watch?v=fss6CrmQU2Y -->

1. Building a basic NL2SQL model
2. Adding few-shot examples
3. Dynamic few-shot example selection
4. Dynamic relevant table selection (Large DB Isssue)
5. Customizing prompts
6. Adding memory to the chatbot so that it answers follow-up questions related to the database.

### Download Dataset
- [Product Dataset](https://www.mysqltutorial.org/getting-started-with-mysql/mysql-sample-database/)

In [ ]:
# !wget https://www.mysqltutorial.org/wp-content/uploads/2023/10/mysqlsampledatabase.zip
# !unzip mysqlsampledatabase.zip
# !rm mysqlsampledatabase.zip

### DB connection

In [ ]:
# check and setup environment variables and API keys for Langchain and OpenAI

import os
import dotenv

dotenv.load_dotenv()

# LANGCHAIN_PROJECT to set the project name for Langchain
print("LANGCHAIN_PROJECT:", os.environ.get("LANGCHAIN_PROJECT", "natural-language-2-sql"))
# LANGCHAIN_TRACING to keep track of the execution
print("LANGCHAIN_TRACING:", os.environ.get("LANGCHAIN_TRACING_V2", "Not set"))
# LANGSMITH_ENDPOINT to set the endpoint for Langsmith
print("Langsmith Endpoint:", os.environ.get("LANGSMITH_ENDPOINT", "Not set"))
# OPENAI_API_KEY to set the API key for OpenAI
print("OpenAI API Key:", os.environ.get("OPENAI_API_KEY", "No API Key found")[:10])
# LANGCHAIN_API_KEY to set the API key for Langchain
print("Langchain API Key:", os.environ.get("LANGCHAIN_API_KEY", "No API Key found")[:10])

In [ ]:
# from docker compose
db_user = "root"
db_password = "root"
db_host = "localhost"
db_name = "classicmodels"


### Building a basic NL2SQL model

In [ ]:
from langchain_community.utilities.sql_database import SQLDatabase

# db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}",sample_rows_in_table_info=1,include_tables=['customers','orders'],custom_table_info={'customers':"customer"})
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}")


In [ ]:
print(f"{db.dialect = }")
print(f"{db.get_usable_table_names() = }")
# print(f"{db.table_info=}")


In [ ]:
# utility function to clean SQL queries

import re


def clean_sql_query(text: str) -> str:
    """
    Clean SQL query by removing code block syntax, various SQL tags, backticks,
    prefixes, and unnecessary whitespace while preserving the core SQL query.

    Args:
        text (str): Raw SQL query text that may contain code blocks, tags, and backticks

    Returns:
        str: Cleaned SQL query
    """
    # Step 1: Remove code block syntax and any SQL-related tags
    # This handles variations like ```sql, ```SQL, ```SQLQuery, etc.
    block_pattern = r"```(?:sql|SQL|SQLQuery|mysql|postgresql)?\s*(.*?)\s*```"
    text = re.sub(block_pattern, r"\1", text, flags=re.DOTALL)

    # Step 2: Handle "SQLQuery:" prefix and similar variations
    # This will match patterns like "SQLQuery:", "SQL Query:", "MySQL:", etc.
    prefix_pattern = r"^(?:SQL\s*Query|SQLQuery|MySQL|PostgreSQL|SQL)\s*:\s*"
    text = re.sub(prefix_pattern, "", text, flags=re.IGNORECASE)

    # Step 3: Extract the first SQL statement if there's random text after it
    # Look for a complete SQL statement ending with semicolon
    sql_statement_pattern = r"(SELECT.*?;)"
    sql_match = re.search(sql_statement_pattern, text, flags=re.IGNORECASE | re.DOTALL)
    if sql_match:
        text = sql_match.group(1)

    # Step 4: Remove backticks around identifiers
    text = re.sub(r"`([^`]*)`", r"\1", text)

    # Step 5: Normalize whitespace
    # Replace multiple spaces with single space
    text = re.sub(r"\s+", " ", text)

    # Step 6: Preserve newlines for main SQL keywords to maintain readability
    keywords = [
        "SELECT",
        "FROM",
        "WHERE",
        "GROUP BY",
        "HAVING",
        "ORDER BY",
        "LIMIT",
        "JOIN",
        "LEFT JOIN",
        "RIGHT JOIN",
        "INNER JOIN",
        "OUTER JOIN",
        "UNION",
        "VALUES",
        "INSERT",
        "UPDATE",
        "DELETE",
    ]

    # Case-insensitive replacement for keywords
    pattern = "|".join(r"\b{}\b".format(k) for k in keywords)
    text = re.sub(f"({pattern})", r"\n\1", text, flags=re.IGNORECASE)

    # Step 7: Final cleanup
    # Remove leading/trailing whitespace and extra newlines
    text = text.strip()
    text = re.sub(r"\n\s*\n", "\n", text)

    return text


In [ ]:
from langchain.chains import create_sql_query_chain
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
generate_query = create_sql_query_chain(llm, db)
query = generate_query.invoke({"question": "what is price of `1968 Ford Mustang`"})
# "what is price of `1968 Ford Mustang`"
query = clean_sql_query(query)
print(query)


In [ ]:
from langchain_community.tools import QuerySQLDataBaseTool


execute_query = QuerySQLDataBaseTool(db=db)
execute_query.invoke(query)


In [ ]:
# chain all steps together

from langchain_core.runnables import RunnablePassthrough, RunnableLambda

chain = generate_query | RunnableLambda(clean_sql_query) | execute_query

chain.invoke({"question": "How many orders are there?"})

In [ ]:
# Print the prompt used in the chain
chain.get_prompts()[0].pretty_print()

In [ ]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result, answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
Answer: """
)

rephrase_answer = answer_prompt | llm | StrOutputParser()

chain = (
    RunnablePassthrough.assign(
        query=generate_query | RunnableLambda(clean_sql_query)
    ).assign(result=itemgetter("query") | execute_query)
    | rephrase_answer
)

chain.invoke({"question": "How many orders are there"})

### Adding few-shot examples

In [ ]:
examples = [
    {
        "input": "List all customers in France with a credit limit over 20,000.",
        "query": "SELECT * FROM customers WHERE country = 'France' AND creditLimit > 20000;",
    },
    {
        "input": "Get the highest payment amount made by any customer.",
        "query": "SELECT MAX(amount) FROM payments;",
    },
    {
        "input": "Show product details for products in the 'Motorcycles' product line.",
        "query": "SELECT * FROM products WHERE productLine = 'Motorcycles';",
    },
    {
        "input": "Retrieve the names of employees who report to employee number 1002.",
        "query": "SELECT firstName, lastName FROM employees WHERE reportsTo = 1002;",
    },
    {
        "input": "List all products with a stock quantity less than 7000.",
        "query": "SELECT productName, quantityInStock FROM products WHERE quantityInStock < 7000;",
    },
    {
        "input": "what is price of `1968 Ford Mustang`",
        "query": "SELECT `buyPrice`, `MSRP` FROM products  WHERE `productName` = '1968 Ford Mustang' LIMIT 1;",
    },
]

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    FewShotChatMessagePromptTemplate,
)

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}\nSQLQuery:"),
        ("ai", "{query}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    # input_variables=["input","top_k"],
    input_variables=["input"],
)
print(few_shot_prompt.format(input1="How many products are there?"))

### Dynamic few-shot example selection

In [ ]:
from langchain_chroma import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_openai import OpenAIEmbeddings

vectorstore = Chroma()
vectorstore.delete_collection()
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    OpenAIEmbeddings(),
    vectorstore,
    k=2,
    input_keys=["input"],
)
example_selector.select_examples({"input": "how many employees we have?"})
# example_selector.select_examples({"input": "How many employees?"})

In [ ]:
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    input_variables=["input", "top_k"],
)
print(few_shot_prompt.format(input="How many products are there?"))

### Customizing prompts

In [ ]:
final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a MySQL expert. Given an input question, create a syntactically correct MySQL query to run. Unless otherwise specificed.\n\nHere is the relevant table info: {table_info}\n\nBelow are a number of examples of questions and their corresponding SQL queries.",
        ),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)
print(
    final_prompt.format(
        input="How many products are there?", table_info="some table info"
    )
)

In [ ]:
generate_query = create_sql_query_chain(llm, db, final_prompt)

chain = (
    RunnablePassthrough.assign(
        query=generate_query | RunnableLambda(clean_sql_query)
    ).assign(result=itemgetter("query") | execute_query)
    | rephrase_answer
)

chain.invoke({"question": "How many csutomers with credit limit more than 50000"})

### Dynamic relevant table selection

In [ ]:
from operator import itemgetter

# from langchain.chains.openai_tools import create_extraction_chain_pydantic
from pydantic import BaseModel, Field
from typing import List


class Table(BaseModel):
    """Table in SQL database."""

    name: List[str] = Field(description="List of Name of tables in SQL database.")


table_details = """
Table Name:productlines
Table Description:Stores information about the different product lines offered by the company, including a unique name, textual description, HTML description, and image. Categorizes products into different lines.

Table Name:products
Table Description:Contains details of each product sold by the company, including code, name, product line, scale, vendor, description, stock quantity, buy price, and MSRP. Linked to the productlines table.

Table Name:offices
Table Description:Holds data on the company's sales offices, including office code, city, phone number, address, state, country, postal code, and territory. Each office is uniquely identified by its office code.

Table Name:employees
Table Description:Stores information about employees, including number, last name, first name, job title, contact info, and office code. Links to offices and maps organizational structure through the reportsTo attribute.

Table Name:customers
Table Description:Captures data on customers, including customer number, name, contact details, address, assigned sales rep, and credit limit. Central to managing customer relationships and sales processes.

Table Name:payments
Table Description:Records payments made by customers, tracking the customer number, check number, payment date, and amount. Linked to the customers table for financial tracking and account management.

Table Name:orders
Table Description:Details each sales order placed by customers, including order number, dates, status, comments, and customer number. Linked to the customers table, tracking sales transactions.

Table Name:orderdetails
Table Description:Describes individual line items for each sales order, including order number, product code, quantity, price, and order line number. Links orders to products, detailing the items sold.
"""
print(table_details)

In [ ]:
table_details_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Return the names of ALL the SQL tables that MIGHT be relevant to the user question.
                          The tables are:

                          {table_details}

                          Remember to include ALL POTENTIALLY RELEVANT tables, even if you're not sure that they're needed.""",
        ),
        ("human", "{question}"),
    ]
)

structured_llm = llm.with_structured_output(Table, method="function_calling")

table_chain = table_details_prompt | structured_llm
tables = table_chain.invoke(
    {
        "question": "give me details of customer and their order count",
        "table_details": table_details,
    }
)
tables

In [ ]:
def get_tables(table_response: Table) -> List[str]:
    """
    Extracts the list of table names from a Table object.

    Args:
        table_response (Table): A Pydantic Table object containing table names.

    Returns:
        List[str]: A list of table names.
    """
    return table_response.name


select_table = (
    {"question": itemgetter("question"), "table_details": itemgetter("table_details")}
    | table_chain
    | get_tables
)
select_table.invoke(
    {
        "question": "give me details of customer and their order count",
        "table_details": table_details,
    }
)

In [ ]:
chain = (
    RunnablePassthrough.assign(table_names_to_use=select_table)
    | RunnablePassthrough.assign(
        query=generate_query | RunnableLambda(clean_sql_query)
    ).assign(result=itemgetter("query") | execute_query)
    | rephrase_answer
)

chain.invoke(
    {
        "question": "How many cutomers with order count more than 5",
        "table_details": table_details,
    }
)

In [ ]:
chain.invoke({"question": "Can you list their names?", "table_details": table_details})

### Adding memory to the chatbot so that it answers follow-up questions related to the database.


In [ ]:
final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a MySQL expert. Given an input question, create a syntactically correct MySQL query to run. Unless otherwise specified.\n\nHere is the relevant table info: {table_info}\n\nBelow are a number of examples of questions and their corresponding SQL queries. Those examples are just for reference and hsould be considered while answering follow up questions",
        ),
        few_shot_prompt,
        MessagesPlaceholder(variable_name="messages"),
        ("human", "{input}"),
    ]
)
print(
    final_prompt.format(
        input="How many products are there?", table_info="some table info", messages=[]
    )
)

In [ ]:
from langchain.memory import ChatMessageHistory

history = ChatMessageHistory()

generate_query = create_sql_query_chain(llm, db, final_prompt)

chain = (
    RunnablePassthrough.assign(table_names_to_use=select_table)
    | RunnablePassthrough.assign(
        query=generate_query | RunnableLambda(clean_sql_query)
    ).assign(result=itemgetter("query") | execute_query)
    | rephrase_answer
)


In [ ]:
question = "How many cutomers with order count more than 5"
response = chain.invoke(
    {"question": question, "messages": history.messages, "table_details": table_details}
)
response

In [ ]:
history.add_user_message(question)
history.add_ai_message(response)


In [ ]:
history.messages

In [ ]:
response = chain.invoke(
    {
        "question": "Can you list there names?",
        "messages": history.messages,
        "table_details": table_details,
    }
)
response